In [ ]:
# @title 🚀 Setup (Run this cell first!)
import os
import sys

# 1. Detect if we are in Google Colab
if 'google.colab' in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # 2. Define the source of your data files 
    BASE_URL = "https://raw.githubusercontent.com/fhfarnoud/intro2ml/main/2026S/data/"
    
    # 3. List the files you need
    files_to_download = ['canterville_ghost.txt', 'emma.txt', 'hamlet.txt', 'bible_kjv.txt']
    
    # 4. Download the files
    if not os.path.exists('data'):
        os.makedirs('data')
        
    for filename in files_to_download:
        if not os.path.exists(f"data/{filename}"):
            url = f"{BASE_URL}{filename}"
            print(f"Downloading {filename}...")
            !wget -q -O data/{filename} {url}
            
    print("✅ Setup complete! Data files are ready.")
else:
    print("Running locally. Assuming data is already present.")



**🤖 AI Lab Partner Policy: STRICTLY Opt-In Code Generation**

In this course, we treat AI tools (like ChatGPT, Gemini, Copilot) as **Lab Partners**, not solution generators. You must use the following prompt to ensure the AI acts responsibly.

**1. Copy the text inside the block below**
**2. Open your AI Assistant (Gemini, ChatGPT, etc.)**
**3. Paste the text to set the rules for the session**

> "I am a student in an Intro to Machine Learning course. Please act as my **ML Lab Partner**.
> 
> **Your Rules:**
> 
> 1. **Code Generation is STRICTLY Opt-In:** You **MUST NOT** generate any runnable Python code unless my message starts with one of the specific prefixes below (`code:` or `output:`).
>    * *Default Behavior:* If I ask 'How do I...?' or 'Help me with...', explain the strategy in English, provide pseudocode, or use illustrative examples. Do not generate runnable solution code.
> 
> 2. **The 'code:' Trigger (Logic & Calculation):** 
>    * When generating code, prioritize simplicity and human readability. Avoid complex syntax.
>    * **Constraint:** When I use this trigger, provide **only one single line of code**. Do not write full blocks.
> 
> 3. **The 'output:' Trigger (Formatting & Printing):**
>    * Use this ONLY when I request code to print results, format tables, or create plots.
>    * **Exception:** For this trigger only, you **MAY** provide full multi-line code blocks to handle the verbose syntax of formatting or plotting.
> 
> 4. **Wait for Me:** After providing the code, stop immediately. Wait for me to run it and ask for the next step.
> 
> 5. **Explain Briefly:** Add a short comment explaining what the code does.
> 
> 6. **Catch Logic Errors:** If I ask for a step that is methodologically wrong (like testing on training data), stop me and explain the error before proceeding."


In [ ]:
import random
import matplotlib.pyplot as plt
from collections import Counter

# Lecture 18: Random Processes & Markov Text Generation

In L17, we generated fake **words** by sampling letters one at a time. Conditioning on the previous letter made the results more English-like.

Today, we scale up from **letters** to **words** and build a text generator from classic literature.

## 1. Word Frequencies in a Book

Let's start with the same text from L17 --- Oscar Wilde's *The Canterville Ghost* --- but now we'll work with **words** instead of letters.

### Recall: `Counter()`

```python
from collections import Counter

words = ["the", "cat", "the", "hat"]
word_counts = Counter(words)              # Counter({'the': 2, 'cat': 1, 'hat': 1})
word_counts.most_common(2)                # [('the', 2), ('cat', 1)]
```

In [ ]:
# Load the full text of The Canterville Ghost
with open("data/canterville_ghost.txt") as f:
    raw_text = f.read()

# Convert to lowercase and split into words
all_words = raw_text.lower().split()
print(f"Total words: {len(all_words)}")
print(f"First 20 words: {all_words[:20]}")

In [ ]:
# Count word frequencies
# YOUR CODE HERE
# TODO: Use Counter to count how often each word appears
word_counts = Counter(...)
raise NotImplementedError()

# Show the 15 most common words
top15 = word_counts.most_common(15)
for word, count in top15:
    print(f"  '{word}': {count} times ({count/len(all_words)*100:.1f}%)")

## 2. Generating Text Independently (No Context)

Our first attempt: pick each word based on how common it is, ignoring all context. This is the word-level version of L17's independent letter sampling.

In [ ]:
# Build the vocabulary and probability list
vocab = list(word_counts.keys())

# Map each word to its index
word_to_index = {}
for i in range(len(vocab)):
    word_to_index[vocab[i]] = i

# probs[j] = probability of vocab[j] overall
probs = []
for w in vocab:
    probs.append(word_counts[w] / len(all_words))

print(f"Vocabulary size: {len(vocab)} unique words")

In [ ]:
# Generate 20 words independently using cumulative sampling
random.seed(42)
independent_words = []
for w in range(20):
    r = random.random()
    cumulative = 0
    for j in range(len(vocab)):
        cumulative = cumulative + probs[j]
        if r < cumulative:
            independent_words.append(vocab[j])
            break

print("Independent (each word drawn from probs):")
print(" ".join(independent_words))

Every word is a real English word, but the result is **gibberish**. There is no structure because we ignored what came before each word.

## 3. What Word Follows What?

To do better, we need **conditional probabilities**: given the current word, what words are likely to come next?

### 3.1 Building the transition table

In [ ]:
# Count how often each word follows each other word
pair_counts = {}
for i in range(len(all_words) - 1):
    current = all_words[i]
    next_word = all_words[i + 1]
    if current not in pair_counts:
        pair_counts[current] = {}
    if next_word not in pair_counts[current]:
        pair_counts[current][next_word] = 0
    pair_counts[current][next_word] = pair_counts[current][next_word] + 1

# Build the conditional probability table as a list of lists
# trans_probs[i][j] = probability of vocab[j] following vocab[i]
# YOUR CODE HERE
# TODO: For each word i in vocab, compute the probability of each next word j
# Hint: look up pair_counts[vocab[i]], divide each count by the row total
trans_probs = []
for i in range(len(vocab)):
    row = [0] * len(vocab)
    ...
    trans_probs.append(row)
raise NotImplementedError()

print(f"trans_probs: {len(trans_probs)} x {len(trans_probs[0])} table")

### 3.2 Examining conditional probabilities

Let's look at what words are most likely to follow specific words.

In [ ]:
# Helper: get the count from a (word, prob) pair, for sorting
def get_prob(pair):
    return pair[1]

# Show top 5 next words for a few example words
for target in ['the', 'was', 'she', 'ghost']:
    t_index = word_to_index[target]
    # Collect non-zero probabilities
    followers = []
    for j in range(len(vocab)):
        if trans_probs[t_index][j] > 0:
            followers.append((vocab[j], trans_probs[t_index][j]))
    followers.sort(key=get_prob, reverse=True)
    
    top5_str = ""
    for w, p in followers[:5]:
        top5_str = top5_str + f"'{w}' ({p*100:.0f}%), "
    print(f"After '{target}': {top5_str[:-2]}")

Every word has a different distribution of next words. After "the", you see nouns and adjectives. After "she", you see verbs. The context completely changes the prediction.

This table of "what follows what" is the core of a **Markov model**.

## 4. Markov Text Generation

A **Markov model** uses only the current word to predict the next one:
1. Start with a word
2. Look up the row in `trans_probs` for that word
3. Sample the next word using cumulative probabilities
4. Repeat

### 4.1 Generating from Canterville Ghost

In [ ]:
# Generate a sentence using the Markov model
random.seed(10)

def generate_sentence(start_word, n_words):
    """Generate n_words starting from start_word using trans_probs."""
    result = [start_word]
    current_index = word_to_index[start_word]
    
    for i in range(n_words - 1):
        # Sample next word from trans_probs[current_index]
        r = random.random()
        cumulative = 0
        chosen_index = 0
        for j in range(len(vocab)):
            cumulative = cumulative + trans_probs[current_index][j]
            if r < cumulative:
                chosen_index = j
                break
        
        result.append(vocab[chosen_index])
        current_index = chosen_index
    
    return " ".join(result)

# Generate from several starting words
for start in ['the', 'she', 'it', 'my']:
    text = generate_sentence(start, 25)
    print(f'Starting with "{start}":')
    print(f'  {text}')
    print()

Much better than the independent model! The text has local coherence --- short phrases that sound like English. But it still lacks long-range structure: sentences don't stay on a single topic.

**TODO:** Compare the independent output (Section 2) with the Markov output above. Why do some 2-word phrases in the Markov output sound natural (like "the ghost") while the independent model never produces them?

**Answer:**

### 4.2 Comparing sources

Different books produce different-sounding text. Let's build Markov models from three classic books and compare.

In [ ]:
# Build a Markov model from a text file and generate a sentence
def build_and_generate(filepath, start_word, n_words):
    with open(filepath) as f:
        text = f.read()
    words = text.lower().split()
    
    # Count word pairs
    counts = {}
    for i in range(len(words) - 1):
        curr = words[i]
        nxt = words[i + 1]
        if curr not in counts:
            counts[curr] = {}
        if nxt not in counts[curr]:
            counts[curr][nxt] = 0
        counts[curr][nxt] = counts[curr][nxt] + 1
    
    # Generate
    result = [start_word]
    current = start_word
    for i in range(n_words - 1):
        if current not in counts:
            break
        candidates = list(counts[current].keys())
        candidate_counts = list(counts[current].values())
        total = sum(candidate_counts)
        
        r = random.random()
        cumulative = 0
        for j in range(len(candidates)):
            cumulative = cumulative + candidate_counts[j] / total
            if r < cumulative:
                current = candidates[j]
                break
        result.append(current)
    
    return " ".join(result)

random.seed(42)
sources = [
    ("data/emma.txt", "Jane Austen - Emma"),
    ("data/hamlet.txt", "Shakespeare - Hamlet"),
    ("data/bible_kjv.txt", "King James Bible"),
]

for filepath, title in sources:
    text = build_and_generate(filepath, "the", 25)
    print(f"{title}:")
    print(f"  {text}")
    print()

Each model captures the **style** of its source --- just by counting word pairs! Austen sounds formal, Shakespeare sounds dramatic, and the Bible sounds archaic. The Markov model doesn't understand meaning; it just learned which words tend to follow which.

### 4.3 Higher-order models

An order-1 model looks at only the previous word. What if we look at the previous **2 words** (order 2)? Or 3? Or 5?

In [ ]:
def build_higher_order(words, order):
    """Build a Markov model that conditions on the previous 'order' words."""
    model = {}
    for i in range(len(words) - order):
        key = tuple(words[i:i + order])
        next_word = words[i + order]
        if key not in model:
            model[key] = {}
        if next_word not in model[key]:
            model[key][next_word] = 0
        model[key][next_word] = model[key][next_word] + 1
    return model

def generate_higher_order(model, order, n_words):
    """Generate text from a higher-order Markov model."""
    # Start with a random key from the model
    keys = list(model.keys())
    start_key = keys[random.randint(0, len(keys) - 1)]
    result = list(start_key)
    
    for i in range(n_words - order):
        key = tuple(result[-order:])
        if key not in model:
            break
        
        candidates = list(model[key].keys())
        candidate_counts = list(model[key].values())
        total = sum(candidate_counts)
        
        r = random.random()
        cumulative = 0
        for j in range(len(candidates)):
            cumulative = cumulative + candidate_counts[j] / total
            if r < cumulative:
                result.append(candidates[j])
                break
    
    return " ".join(result)

# Compare different orders using Emma
with open("data/emma.txt") as f:
    emma_words = f.read().lower().split()

random.seed(7)
for order in [1, 2, 3, 5]:
    model = build_higher_order(emma_words, order)
    text = generate_higher_order(model, order, 25)
    print(f"Order {order}:")
    print(f"  {text}")
    print()

**TODO:** As we increase the Markov order from 1 to 5, the generated text gets more coherent but also starts to look like it's copied from the book. Why does an order-5 model with a single book mostly reproduce the source verbatim?

**Answer:**

## Summary

1. Generating words **independently** produces gibberish --- word order matters
2. A **Markov model** predicts the next word from the current word (order 1) or recent words (order $k$)
3. Building the model = counting word pairs in a text. Generating = sampling from conditional probabilities.
4. Different books produce different styles --- the model captures patterns without understanding meaning
5. Higher order = more coherent but needs more data, and risks memorizing the source
6. **Limitation**: no long-range coherence, no understanding of meaning

**Next time (L19):** Word embeddings --- representing meaning as vectors, and the ideas behind large language models.